# Case Study: Search Ranking & Ads CTR

Search and ads are the highest-revenue ML systems at most large tech companies. This note covers query understanding, lexical + semantic hybrid retrieval, learning-to-rank, and ads-specific CTR calibration for auctions.

## What Interviewers Test
- Query understanding pipeline: spelling, intent, entity extraction
- Hybrid retrieval: BM25 + semantic, why you need both
- LTR: pointwise vs pairwise vs listwise — when to use each
- CTR calibration: why it matters for auctions and how to do it
- Position bias in search result pages

## Search System Architecture

```
Query → [Understand] → [Retrieve] → [Rank] → [Ads inject] → Results

Understand: spell correction, query expansion, intent classification, NER
Retrieve:   BM25 (lexical) + ANN (semantic) → merge candidates
Rank:       LTR model (GBDT or neural) on rich features
Ads:        CTR model → calibrated bid → auction → inject at positions
```


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

# --- BM25 implementation ---
def bm25_score(query_terms, doc_terms, idf, avg_dl, k1=1.5, b=0.75):
    """BM25 relevance score for a single (query, document) pair."""
    dl = len(doc_terms)
    tf_doc = {}
    for t in doc_terms:
        tf_doc[t] = tf_doc.get(t, 0) + 1
    score = 0.0
    for term in query_terms:
        if term in tf_doc:
            tf = tf_doc[term]
            idf_t = idf.get(term, 0)
            score += idf_t * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avg_dl))
    return score

# Mini corpus
corpus = {
    'd1': 'machine learning model training optimization',
    'd2': 'deep learning neural network architecture',
    'd3': 'machine learning interview preparation guide',
    'd4': 'cooking recipes and food preparation',
    'd5': 'optimization algorithms for machine learning',
}

# Compute IDF
from collections import Counter
import math
N = len(corpus)
doc_terms = {did: text.split() for did, text in corpus.items()}
df = Counter(t for toks in doc_terms.values() for t in set(toks))
idf = {t: math.log((N - df[t] + 0.5) / (df[t] + 0.5) + 1) for t in df}
avg_dl = np.mean([len(t) for t in doc_terms.values()])

query = 'machine learning optimization'
bm25_scores = {did: bm25_score(query.split(), toks, idf, avg_dl)
               for did, toks in doc_terms.items()}
ranked = sorted(bm25_scores.items(), key=lambda x: -x[1])
print("BM25 ranking for 'machine learning optimization':")
for did, score in ranked:
    print(f"  {did}: {score:.3f}  |  {corpus[did][:50]}")


## Learning-to-Rank (LTR)

| Approach | What it optimizes | Loss | Use when |
|---|---|---|---|
| **Pointwise** | Per-document relevance score | MSE, cross-entropy | Simple, fast baseline |
| **Pairwise** | Relative order of document pairs | Pairwise hinge/log loss | Better NDCG than pointwise |
| **Listwise** | Full list ordering | LambdaRank, ListNet | Best NDCG; expensive |

Most production systems use **LambdaMART** (GBDT + pairwise gradient) or a neural listwise ranker.


In [ ]:
# --- Pairwise loss implementation ---
def pairwise_loss(scores, labels):
    """
    Simplified RankNet pairwise cross-entropy loss.
    scores: (n,) predicted scores for documents
    labels: (n,) relevance labels
    """
    n = len(scores)
    total_loss = 0.0
    n_pairs = 0
    for i in range(n):
        for j in range(n):
            if labels[i] > labels[j]:
                # i should rank above j
                prob_ij = 1 / (1 + np.exp(-(scores[i] - scores[j])))
                total_loss -= np.log(prob_ij + 1e-12)
                n_pairs += 1
    return total_loss / max(n_pairs, 1)

# Test
scores = np.array([0.9, 0.3, 0.7, 0.1, 0.5])
labels = np.array([2,   0,   1,   0,   1])    # relevance grades
loss = pairwise_loss(scores, labels)
print(f"Pairwise loss (good ranking): {loss:.4f}")

# Reversed ranking (bad)
loss_bad = pairwise_loss(-scores, labels)
print(f"Pairwise loss (bad ranking):  {loss_bad:.4f}")


## CTR Calibration for Ads Auctions

In a second-price auction: **bid = CTR × value_per_click**

If CTR is miscalibrated:
- Over-estimated CTR → over-bid → revenue for platform but buyer overpays
- Under-estimated CTR → under-bid → missed win, lost revenue

**Platt scaling:** Fit a logistic regression on the model's raw scores using held-out data. Simple and effective.


In [ ]:
# --- CTR calibration demo ---
np.random.seed(42)
n = 3000
y_true = (np.random.rand(n) < 0.1).astype(int)  # 10% CTR

# Simulate an overconfident model (scores too extreme)
y_score_uncal = np.clip(
    np.where(y_true == 1, np.random.beta(8, 1, n), np.random.beta(1, 8, n)),
    0.01, 0.99
)

# Platt scaling calibration
from sklearn.model_selection import train_test_split
X_cal_tr, X_cal_te, y_cal_tr, y_cal_te = train_test_split(
    y_score_uncal.reshape(-1,1), y_true, test_size=0.4, random_state=42
)
platt = LogisticRegression().fit(X_cal_tr, y_cal_tr)
y_cal = platt.predict_proba(X_cal_te.reshape(-1,1))[:,1]
y_uncal_te = X_cal_te.ravel()

# Compare calibration
def ece(y_true, y_score, n_bins=10):
    bins = np.linspace(0,1,n_bins+1)
    ece_val = 0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_score>=lo) & (y_score<hi)
        if not mask.any(): continue
        ece_val += mask.mean() * abs(y_true[mask].mean() - y_score[mask].mean())
    return ece_val

print(f"ECE before calibration: {ece(y_cal_te, y_uncal_te):.4f}")
print(f"ECE after Platt scaling: {ece(y_cal_te, y_cal):.4f}")
print(f"Mean predicted CTR (uncal): {y_uncal_te.mean():.4f}, true: {y_cal_te.mean():.4f}")
print(f"Mean predicted CTR (cal):   {y_cal.mean():.4f}, true: {y_cal_te.mean():.4f}")


## Common Interview Questions

**Q: Why use BM25 and semantic retrieval together (hybrid)?**
BM25 excels at keyword matching — it finds documents with exact query terms. Semantic retrieval (dense embeddings) captures meaning — it finds related documents even without term overlap. Hybrid retrieval handles both exact matches ("AAPL stock price") and semantic queries ("is apple doing well financially"), covering the full query distribution.

**Q: What is position bias in search and how do you correct it?**
Results shown at higher positions receive more clicks due to position, not quality. This biases CTR training data to inflate the quality of top results. Correct with inverse propensity weighting (weight clicks by 1/position_propensity), or with a position-aware model that takes position as an input and removes it at serving.

**Q: Why does CTR calibration matter for auctions?**
In second-price auctions, each advertiser should bid CTR × value. If CTR is over-estimated, advertisers overbid and may lose money or bid budget. If under-estimated, they under-bid and lose impressions. Platform revenue depends on accurate CTR estimates because the auction ranking is by expected value (CTR × bid). Miscalibration directly affects revenue allocation.

**Q: What is LambdaMART?**
LambdaMART is gradient boosted trees trained with LambdaRank gradients. LambdaRank defines pairwise gradients that are weighted by the change in NDCG from swapping the pair — this directly optimizes the ranking metric. LambdaMART is the standard LTR algorithm in industry (used in most search/ads systems as the GBDT ranker).

## Key Takeaways
- Search pipeline: query understanding → hybrid retrieval (BM25 + ANN) → LTR ranker → ads injection
- Hybrid retrieval: BM25 for exact matches, dense embeddings for semantic matches
- LTR: pointwise (easy) → pairwise (better NDCG) → listwise (best, expensive)
- CTR calibration via Platt scaling or isotonic regression before auction systems
- Position bias: correct with IPW or position-aware models; never train naively on position-biased clicks
- LambdaMART: GBDT with NDCG-weighted pairwise gradients — production standard for search ranking